# glmhmm quickstart

Switching linear-Gaussian regression (input-driven HMM) for multi-fiber photometry.

Part 1 runs the synthetic demo to verify the install and show what recovered structure looks like.
Part 2 is the real-data template — edit the marked cell and run.

## Part 1 — synthetic demo

Known planted structure: K=2 in `VMH (E)` and `PrL (E)` (state 0 weights resident kinematics, state 1 weights intruder), state-independent drive in `NAc (DA)`, event-only responses elsewhere. ~2 min.

In [ ]:
import matplotlib
import glmhmm
from glmhmm import *

out = glmhmm.demo.run()

### Figures from the demo fit

In [ ]:
set_nature_rc()
ss, m, allr, summ = out['sessionset'], out['model'], out['loao_all'], out['K_summary']

fig, _ = plot_K_selection(allr, summ, shuffle=out.get('shuffle'))
fig, _ = plot_loao_r2(allr, out['best_K'], ss.regions)
fig, _ = plot_state_raster(m, ss, session_index=0)
fig, _ = plot_weight_matrix(out['weights'], 'VMH (E)')
fig, _ = plot_state_behavior(out['correspondence'])

### The null case

`switch_strength=0` removes the switching. K>1 should **not** win. Run this before trusting any K>1 result on real data — it calibrates what "no structure" looks like in these same numbers.

In [ ]:
null = glmhmm.demo.run(Ks=(1, 2), switch_strength=0.0,
                       verbose=False, run_controls=False)
null['K_summary'].round(4)

---
## Part 2 — real data

Load the pickle and the feature/label dictionaries, then edit the cell below.

In [ ]:
import pandas as pd, numpy as np

full_data = pd.read_pickle('multifiber.pickle')
# features, sniffing_labels_dict loaded alongside

full_data = {k.strip(): v for k, v in full_data.items()}
features = {k.strip(): v for k, v in features.items()}
sniffing_labels_dict = {k.strip(): v for k, v in sniffing_labels_dict.items()}

agg_animals = ['3095','3096','3097','4013','4014','4015','4016','91R2','30L','30B','31R2','29L2']
obs_animals = ['30R2','29L','86L','87L2','927R','927L','933R','1185','1162B']
non_animals = ['87R2','87L','87B','86L2','583L2','583B','1162L2','1162R']
GROUP_OF_ANIMAL = ({a: 'EXP' for a in agg_animals}
                   | {a: 'OBS' for a in obs_animals}
                   | {a: 'NON' for a in non_animals})

REGIONS = ['VMH (E)', 'VMH (I)', 'PrL (E)', 'PrL (I)', 'NAc (DA)']

keys = [k for k in sorted(set(full_data) & set(features))
        if (mm := parse_session(k, GROUP_OF_ANIMAL))
        and mm['group'] == 'OBS' and mm['day'] != 9 and mm['cond_norm'] == 'obs']
print(len(keys), 'sessions')

### Audit the condition tokens

`1162B_d9_OBSmCD1_t1` showed the keys are not a two-value condition field. Check the token set before trusting the filter above.

In [ ]:
aud = pd.DataFrame([parse_session(k, GROUP_OF_ANIMAL)
                    for k in sorted(set(full_data) & set(features))
                    if parse_session(k, GROUP_OF_ANIMAL)])
display(aud.query("group == 'OBS'").pivot_table(
    index='cond', columns='day', values='trial', aggfunc='count', fill_value=0))
print('unassigned animals:', sorted(aud.loc[aud.group == 'UNKNOWN', 'animal'].unique()))

### Audit 1 — do the labels share a clock with the neural data?

`delta` must be clearly positive. If it is near zero the label vector is offset or decimated and truncating from frame 0 is wrong — this failure is invisible downstream.

In [ ]:
al = audit_alignment(keys, full_data, sniffing_labels_dict)
display(al.round(3).head(15))
print(al.delta.describe().round(3).to_dict())

### Audit 2 — effective sample size vs design width

More than roughly 1 predictor per 5 effective samples and ridge shrinks the filter bank to zero whatever the penalty.

In [ ]:
dcfg = DesignConfig(fs=40.0, n_basis=5, n_ar=2)
display(audit_effective_n(keys, full_data, features, REGIONS, dcfg).round(1))

### Build

In [ ]:
ss = build_sessionset(keys, full_data, features, sniffing_labels_dict,
                      REGIONS, dcfg, GROUP_OF_ANIMAL)
print(ss.design_report())
display(ss.summary())

### Select K

K=1 is ridge regression. Read `delta_median` (paired per-animal gain over K=1) and `n_improved`, not the raw median.

In [ ]:
allr, summ = select_K(ss, Ks=(1, 2, 3, 4),
                      base=FitConfig(ridge=10.0, n_restarts=3))
display(summ.round(4))

### Fit and inspect

In [ ]:
K = 3   # set from the table above
m = SwitchingGLM(FitConfig(n_states=K, ridge=10.0, n_restarts=5)).fit(ss, verbose=True)

display(state_summary(m, ss).round(3))
display(weight_table(m, ss).round(3))

corr = state_behavior_correspondence(m, ss)
print('ARI(state, behavior) =', round(corr['adjusted_rand'], 3),
      ' -- above ~0.3 and the states are behavior, not encoding')
display(corr['crosstab'].round(3))

### Controls

The shuffle gain is the comparison that matters: a K>1 model finds drift regimes in shuffled data too, so compare the K-vs-K1 **gain**, not absolute LL.

In [ ]:
display(shuffle_control(ss, FitConfig(n_states=K), n_shuffle=5)
        .groupby('K').ll_per_frame.describe().round(4))

display(variance_partition(ss, FitConfig(n_states=K)).round(4))

### Alternative framings to fit before committing

These are different scientific questions, not robustness checks.

In [ ]:
# States as encoding regimes WITHIN engagement
ss_att = build_sessionset(keys, full_data, features, sniffing_labels_dict,
                          REGIONS, dcfg, GROUP_OF_ANIMAL, restrict_to_label=4)

# Without the AR block: AR lags stop the HMM segmenting drift, but on a
# smooth signal they also absorb most predictable variance.
ss_noar = build_sessionset(keys, full_data, features, sniffing_labels_dict,
                           REGIONS, DesignConfig(n_basis=5, n_ar=0),
                           GROUP_OF_ANIMAL)

### Figures

In [ ]:
set_nature_rc()
fig, _ = plot_K_selection(allr, summ);            savefig(fig, 'fig/K_selection')
fig, _ = plot_loao_r2(allr, K, ss.regions);       savefig(fig, 'fig/loao_r2')
fig, _ = plot_state_raster(m, ss, 0);             savefig(fig, 'fig/raster')
fig, _ = plot_weight_matrix(weight_table(m, ss), 'PrL (E)')
savefig(fig, 'fig/weights_PrL_E')
fig, _ = plot_state_behavior(corr);               savefig(fig, 'fig/state_behavior')